In [ ]:
import zipfile
with zipfile.ZipFile("/data2/qn/MRAMG/MRAMG-Bench/IMAGE.zip", "r") as z:
    print(z.namelist())

In [2]:
extract_dir = "/data2/qn/MRAMG/MRAMG-Bench/IMAGE"

with zipfile.ZipFile("/data2/qn/MRAMG/MRAMG-Bench/IMAGE.zip", 'r') as z:
    z.extractall(extract_dir)


In [2]:
# 调用bge-m3，生成embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/data2/qn/MRAMG/models/bge-m3")

/data2/qn/anaconda3/envs/kgqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("/data2/qn/MRAMG/models/bge-m3")
sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings = model.encode(sentences)

/data2/qn/anaconda3/envs/kgqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [42]:
from llama_index.core.node_parser import SentenceSplitter
# 初始化切片器
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=0)
import re
def get_chunks_with_metadata(doc, meta_base):
    """
    doc: (doc_id, text, img_list)
    meta_base: 基础元数据，例如 {"source": "manual.pdf"}
    """
    doc_id, text, img_list = doc
    
    # --- 步骤 1: 预处理 - 给 <PIC> 加上索引编号 ---
    # 变成 <PIC_0>, <PIC_1> ... 
    # 由于后面有空格，splitter 几乎百分之百会完整保留这个标签
    counter = 0
    def tag_indexer(match):
        nonlocal counter
        res = f"<PIC_{counter}>"
        counter += 1
        return res
    
    indexed_text = re.sub(r"<PIC>", tag_indexer, text)

    # --- 步骤 2: 切分文本 ---
    # 假设你已经初始化了 splitter = SentenceSplitter(...)
    text_chunks = splitter.split_text(indexed_text)
    
    final_data = []
    for i, chunk in enumerate(text_chunks):
        # --- 步骤 3: 提取当前 chunk 包含的所有图片索引 ---
        # 寻找形如 <PIC_n> 的内容
        found_indices = re.findall(r"<PIC_(\d+)>", chunk)
        
        # 根据索引从 img_list 中捞取对应的图片
        # 即使两个 chunk 有重叠，它们也会根据编号各自捞取正确的图片
        current_chunk_imgs = [img_list[int(idx)] for idx in found_indices]
        
        # --- 步骤 4: 文本还原 ---
        # 将 <PIC_n> 还原为标准 <PIC>，保持数据纯净
        clean_chunk = re.sub(r"<PIC_\d+>", "<PIC>", chunk)
        
        data = {
            "content": clean_chunk,
            "metadata": {
                **meta_base,
                "doc_id": doc_id,
                "imgs_len_of_doc": len(img_list),
                "chunk_index": i,
                "include_img_list": ",".join(map(str, current_chunk_imgs)) # 这里存的是该 chunk 对应的图片(路径/对象)列表
            },
            "embedding": None # 待后续调用 model.encode
        }
        final_data.append(data)
        
    return final_data

In [52]:
import json
jsonl_file = "/data2/qn/MRAMG/MRAMG-Bench/doc_wit.jsonl"
all_chunks = []
with open(jsonl_file, "r") as f:
    for line in f:
        data = json.loads(line)
        all_chunks.extend(get_chunks_with_metadata(data, {"source": "doc_wit"}))


In [7]:
import chromadb
# 1. 初始化本地持久化客户端
# 数据会保存在当前目录下的 'my_local_db' 文件夹中
client = chromadb.PersistentClient(path="/data2/qn/MRAMG/chromadb")

# 2. 创建或获取 Collection（相当于数据库中的表）
# 注意：Chroma 默认使用自带的 embedding 模型，但我们要用自己的 BGE-M3
collection = client.get_or_create_collection(name="doc_arxiv")

In [54]:
ids = [f"id_{i}" for i in range(len(all_chunks))]  
    # 使用你已加载好的 model 生成 embeddings
texts = [c["content"] for c in all_chunks]
embeddings = model.encode(texts)
for i, emb in enumerate(embeddings):
    all_chunks[i]["embedding"] = emb.tolist()

In [55]:
collection.add(
    ids=ids, 
    embeddings=[c["embedding"] for c in all_chunks],
    metadatas=[c["metadata"] for c in all_chunks],
    documents=[c["content"] for c in all_chunks]
)

In [10]:
# 5. 检索示例
query = "How does MVSplat achieve efficient 3D Gaussian prediction and fast inference?"
query_emb = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_emb,
    n_results=3,
    # 显式指定需要包含的内容：默认只有 metadatas 和 documents
    include=["documents", "metadatas", "distances"] 
)
print(results)

{'ids': [['id_8', 'id_9', 'id_7']], 'embeddings': None, 'documents': [['4.3.Assessing model efficiency. As reported in Tab. 1, apart from attaining su_x0002_perior image quality, MVSplat also shows the fastest inference time among all the compared models, accompanied by a lightweight model size, demonstrat_x0002_ing its efficiency and practical utility. It is noteworthy that the reported time encompasses both image encoding and rendering stages. For an in-depth time comparison with pixelSplat [1], our encoder runs at 0.043s, which is more than 2× faster than pixelSplat (0.102s). Besides, pixelSplat predicts 3 Gaussians per_x0002_pixel, while our MVSplat predicts 1 single Gaussian, which also contributes to our faster rendering speed (0.0015s vs. 0.0025s) due to the threefold reduction in the number of Gaussians. More importantly, equipped with the cost volume_x0002_based encoder, our MVSplat enables fast feed-forward inference of 3D Gaussians with a much light-weight design, resulting 

In [13]:
results['documents'][0][2]

'Unlike pixelSplat [1] that predicts probabilistic depth, we develop an efficient and high-performance multi-view depth estimation model that enables unprojecting predicted depth maps as the Gaussian centers, in parallel with another branch for prediction of other Gaussian parameters (αj , Σj and cj ). Our full model, illustrated in <PIC>, is trained end-to-end using only a simple rendering loss for supervision. Next, we discuss the key components.The qualitative comparisons of the top three best models are visualized in <PIC>. MVSplat achieves the highest quality on novel view results even un_x0002_der challenging conditions, such as these regions with repeated patterns (“win_x0002_dow frames” in 1st row), or these present in only one of the input views (“stair handrail” and “lampshade” in 2nd and 3rd rows), or when depicting large-scale outdoor objects captured from distant viewpoints (“bridge” in 4th row). The baseline methods exhibit obvious artifacts for these regions, while MVSpl

In [23]:
import re
import json
def get_caption_dict(doc_name, img_list):
    with open(f"/data2/qn/MRAMG/MRAMG-Bench/IMAGE/IMAGE/images_info/{doc_name}_imgs_collection.json") as f:
        data = json.load(f)
    return {img: data[img]['image_caption'] for img in img_list}

def build_prompt_from_chroma(doc_name, chunks):
    
    docs = chunks["documents"][0]
    metas = chunks["metadatas"][0]
    
    context_parts = []
    image_captions = []
    
    global_img_id = 1
    img_path_to_id = {}
    
    for i, (text, meta) in enumerate(zip(docs, metas)):
        
        img_str = meta.get("include_img_list", "")
        
        # 如果没有图片
        if not img_str.strip():
            context_parts.append(f"[context_{i+1}] {text}")
            continue
        
        img_list = img_str.split(",")
        caption_dict = get_caption_dict(doc_name, img_list)
        
        # 逐个替换 <PIC>
        for img_path in img_list:
            
            if img_path not in img_path_to_id:
                img_path_to_id[img_path] = global_img_id
                image_captions.append(f"[img{global_img_id}_caption] {caption_dict[img_path]}")
                global_img_id += 1
            
            img_id = img_path_to_id[img_path]
            
            # 替换一个 <PIC>
            text = text.replace("<PIC>", f"<img{img_id}>", 1)
        
        context_parts.append(f"[context_{i+1}] {text}")
    
    context_block = "\n".join(context_parts)
    caption_block = "\n".join(image_captions)
    
    return context_block, caption_block

In [5]:
JUDGE_CONFLICT_TEMPLATES = {
    "set_conflict": """Conflict Type: INCLUSION (Set Conflict)\nDisputed Item: {disputed_image}
The {defender_role} argued to INCLUDE this image, while the {challenger_role} argued to EXCLUDE it.""",

    "order_conflict": """Conflict Type: SEQUENTIAL (Order Conflict)
Disputed Item: Image Ordering
Both agents agreed to include the images, but disagreed on the sequence.
- {defender_role}'s proposed sequence: {defender_img_order}
- {challenger_role}'s proposed sequence: {challenger_img_order}"""
}

In [6]:
JUDGE_CONFLICT_TEMPLATES['order_conflict'].format(defender_role="defender", challenger_role="challenger", defender_img_order="1,2,3", challenger_img_order="3,2,1")

"Conflict Type: SEQUENTIAL (Order Conflict)\nDisputed Item: Image Ordering\nBoth agents agreed to include the images, but disagreed on the sequence.\n- defender's proposed sequence: 1,2,3\n- challenger's proposed sequence: 3,2,1"

In [24]:
build_prompt_from_chroma('arxiv', results)

('[context_1] 4.3.Assessing model efficiency. As reported in Tab. 1, apart from attaining su_x0002_perior image quality, MVSplat also shows the fastest inference time among all the compared models, accompanied by a lightweight model size, demonstrat_x0002_ing its efficiency and practical utility. It is noteworthy that the reported time encompasses both image encoding and rendering stages. For an in-depth time comparison with pixelSplat [1], our encoder runs at 0.043s, which is more than 2× faster than pixelSplat (0.102s). Besides, pixelSplat predicts 3 Gaussians per_x0002_pixel, while our MVSplat predicts 1 single Gaussian, which also contributes to our faster rendering speed (0.0015s vs. 0.0025s) due to the threefold reduction in the number of Gaussians. More importantly, equipped with the cost volume_x0002_based encoder, our MVSplat enables fast feed-forward inference of 3D Gaussians with a much light-weight design, resulting in 10× fewer parameters and more than 2× faster speed comp

In [1]:
prompt = open('prompts/text.txt', 'r').read()
query = "what is your name"
context = "dadaiu dadadad daa"
caption = "['a man is running', 'a man is running']"
llm_input = prompt.format(query=query, context=context, caption=caption[0])


In [13]:
str_json = """```json
{
  "winner": "TEXT_AGENT",
  "resolution": "INCLUDE",
  "verdict_reasoning": "The Text Agent's argument for including <img24> is more aligned with the MRAMG principles of maximizing logical coherence and informative value. The user query specifically asks about locating buttons on the air conditioner remote control, and the **JET** button is an advanced feature prominently described in [context_10]. Including <img24> provides visual clarity for this button, which is essential for quick temperature adjustment—a function users may actively seek. 

The Text Agent's explanation follows a logical narrative flow, starting with basic functions and progressing to advanced features, such as the **JET** button. This structure ensures completeness and relevance to the user's query. Excluding <img24> would omit a key visual reference for an important feature, reducing the response's informativeness and usability.

Additionally, the inclusion of <img24> adheres to the physical and causal constraints of the remote control's functionality, as it directly supports the description of the **JET** button's purpose and placement. The Visual Agent's argument to exclude the image does not sufficiently address the importance of visualizing advanced features for user understanding. Therefore, the Text Agent's proposal better fulfills the multimodal generation principles and ensures a comprehensive response."
"""

In [16]:
from utils import parse_json
import json
import re
import ast
from typing import Any, Dict, Optional


def parse_llm_json(text: str) -> Optional[Dict[str, Any]]:
    def extract_json_candidates(text):
        candidates = []

        # ```json block
        matches = re.findall(r"```json\s*(.*?)\s*```", text, re.S | re.I)
        candidates.extend(matches)

        # ``` block
        matches = re.findall(r"```\s*(.*?)\s*```", text, re.S)
        candidates.extend(matches)

        # {...}
        matches = re.findall(r"\{.*?\}", text, re.S)
        candidates.extend(matches)

        candidates.append(text)

        return candidates


    def clean_json_string(s: str):

        s = s.strip()

        # 去掉尾逗号
        s = re.sub(r",\s*([}\]])", r"\1", s)

        # 修复字符串内部换行
        result = []
        in_string = False

        for c in s:
            if c == '"':
                in_string = not in_string
                result.append(c)
            elif c == "\n" and in_string:
                result.append("\\n")
            else:
                result.append(c)

        return "".join(result)


    candidates = extract_json_candidates(text)

    for cand in candidates:

        cleaned = clean_json_string(cand)

        # 1️⃣ 标准 JSON
        try:
            return json.loads(cleaned)
        except Exception:
            pass

        # 2️⃣ Python dict fallback
        try:
            return ast.literal_eval(cleaned)
        except Exception:
            pass

    return None
res = parse_llm_json(str_json)


In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    # 若没有配置环境变量，请用阿里云百炼API Key将下行替换为：api_key="sk-xxx",
    # 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # 以下为北京地域的 base_url，若使用弗吉尼亚地域模型，需要将base_url换成https://dashscope-us.aliyuncs.com/compatible-mode/v1
    # 若使用新加坡地域的模型，需将base_url替换为：https://dashscope-intl.aliyuncs.com/compatible-mode/v1
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.5-plus", # 此处以qwen3.5-plus为例，可按需更换模型名称。模型列表：https://help.aliyun.com/zh/model-studio/models
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"
                    },
                },
                {"type": "text", "text": "图中描绘的是什么景象?"},
            ],
        },
    ],
)
print(completion.choices[0].message.content)

None


In [ ]:
from openai import OpenAI
from img_server import get_local_image_url
img_file = "/data2/qn/MRAMG/MRAMG-Bench/IMAGE/IMAGE/images/MANUAL/Manual01_23.jpg"
url = get_local_image_url(img_file, "MRAMG-Bench/IMAGE/IMAGE/images", "http://127.0.0.1:8009")
client = OpenAI(
    base_url="http://127.0.0.1:8005/v1",
    api_key="llama",
)

completion = client.chat.completions.create(
    model="/data2/qn/KGQA/models/Qwen2-VL-7B-Instruct",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url
                    },
                },
                {"type": "text", "text": "图中描绘的是什么景象?"},
            ],
        },
    ],
)

print(completion.choices[0].message)

In [1]:
_clip_filter = None
def get_clip_filter():
    """懒加载 CLIP 过滤器"""
    global _clip_filter
    if _clip_filter is None:
        from utils.clip_image_filter import get_clip_filter as _get
        _clip_filter = _get()
    return _clip_filter

In [2]:
clip_filter = get_clip_filter()

/data2/qn/anaconda3/envs/kgqa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
all_img_paths = ['MRAMG-Bench/IMAGE/IMAGE/images/MANUAL/Boat_02.png', 'MRAMG-Bench/IMAGE/IMAGE/images/MANUAL/air_fryer_01.png']
question = 'boat'
all_img_paths = clip_filter.filter_images(question, all_img_paths, 10)

In [9]:
all_img_paths

['MRAMG-Bench/IMAGE/IMAGE/images/MANUAL/Boat_02.png',
 'MRAMG-Bench/IMAGE/IMAGE/images/MANUAL/air_fryer_01.png']

In [14]:
import os
import json
from collections import defaultdict
def get_table_metrics(input_folder_path: str):
    """
    从指定文件夹路径读取 JSONL 文件，统计指标并返回表格格式
    """
    # 定义各类数据集
    datasets = {
        "academic": ["arxiv"],
        "lifestyle": ["manual"],
        "web": ["web", "wiki", "wit"]
    }

    # 获取文件夹下所有 JSONL 文件
    jsonl_files = [f for f in os.listdir(input_folder_path) if f.endswith(".jsonl")]

    # 按文件名去掉后缀，获取文档名（假设文件名格式为 <dataset>_xxx.jsonl）
    jsonl_doc_names = [f.split("_")[0] for f in jsonl_files]

    # 存储统计结果
    results = {}

    # 遍历每个大类
    for category, subdatasets in datasets.items():
        # 检查该类别下所有子数据集文件是否都存在
        if all(sub in jsonl_doc_names for sub in subdatasets):
            # 收集所有指标
            metrics_accumulator = defaultdict(list)
            total_records = 0

            for sub in subdatasets:
                # 找到对应的 JSONL 文件
                file_name = next(f for f in jsonl_files if f.startswith(sub + "_"))
                file_path = os.path.join(input_folder_path, file_name)

                with open(file_path, "r", encoding="utf-8") as f:
                    for line in f:
                        record = json.loads(line.strip())
                        prec = record.get("image_precision", 0.0)
                        rec = record.get("image_recall", 0.0)
                        f1 = record.get("image_f1", 0.0)
                        ord = record.get("image_ordering", 0.0)
                        rl = record.get("rougeLsum_f1", 0.0)
                        bs = record.get("bert_score_f1", 0.0)
                        rel = record.get("eval_image_relevance_score", 0.0)
                        eff = record.get("eval_image_effectiveness_score", 0.0)
                        comp = record.get("eval_answer_quality_score", 0.0)
                        pos = record.get("eval_image_position_score", 0.0)
                        metrics_accumulator["prec"].append(prec)
                        metrics_accumulator["rec"].append(rec)
                        metrics_accumulator["f1"].append(f1)
                        metrics_accumulator["ord"].append(ord)
                        metrics_accumulator["rl"].append(rl)
                        metrics_accumulator["bs"].append(bs)
                        metrics_accumulator["rel"].append(rel / 5)
                        metrics_accumulator["eff"].append(eff / 5)
                        metrics_accumulator["comp"].append(comp / 5)
                        metrics_accumulator["pos"].append(pos)
                        total_records += 1
            # 数值乘100，保留两位小数
            category_metrics = {metric: round((sum(values)/len(values)) * 100, 2) for metric, values in metrics_accumulator.items()}
            # 如果是web或arxiv，category_metrics['avg']为10个指标中除了ord之外的平均值
            if category in ["web", "arxiv"]:
                category_metrics["avg"] = round(sum([category_metrics[metric] for metric in category_metrics if metric != "ord"])/9, 2)
            else:
                category_metrics["avg"] = round(sum([category_metrics[metric] for metric in category_metrics])/10, 2)
                
            results[category] = category_metrics
    
    return results


In [13]:
# input_folder_path = "/data2/qn/MRAMG/outputs/gpt-4o-mini/v2"
input_folder_path = "/data2/qn/MRAMG/outputs/gpt-4o-mini/v2"
results = get_table_metrics(input_folder_path)
print(f"统计文件夹：{input_folder_path}")
for category, metrics in results.items():
    if category in ["academic", "web"]:
        print(category, f"& {metrics['prec']} & {metrics['rec']} & {metrics['rel']} & {metrics['comp']} & {metrics['pos']} & {metrics['avg']}")
    if category == "lifestyle":
        print(category, f"& {metrics['prec']} & {metrics['rec']} & {metrics['rel']} & {metrics['ord']} & {metrics['comp']} & {metrics['pos']} & {metrics['avg']}")

NameError: name 'get_table_metrics' is not defined

In [38]:
input_folder_path = "outputs/gpt-4o-mini/v2/"
# input_folder_path = "outputs/gpt-4o-mini/v2/ablation/clip_top5"
results = get_table_metrics(input_folder_path)
print(f"统计文件夹：{input_folder_path}")
for category, metrics in results.items():
    if category in ["academic", "web"]:
        print(category, f"& {metrics['prec']} & {metrics['rec']} & {metrics['rel']} & {metrics['comp']} & {metrics['pos']} & {metrics['avg']}")
    if category == "lifestyle":
        print(category, f"& {metrics['prec']} & {metrics['rec']} & {metrics['f1']} & {metrics['ord']} & {metrics['rl']} & {metrics['bs']} & {metrics['rel']} & {metrics['eff']} & {metrics['comp']} & {metrics['pos']} & {metrics['avg']}")

统计文件夹：outputs/gpt-4o-mini/v2/
academic & 64.23 & 85.35 & 88.89 & 96.36 & 88.38 & 76.53
lifestyle & 44.42 & 60.19 & 44.87 & 39.02 & 40.86 & 86.05 & 85.08 & 62.28 & 85.91 & 76.42 & 62.51
web & 94.32 & 98.86 & 92.64 & 94.0 & 96.33 & 85.92


In [4]:
import os
import json
import re

from collections import defaultdict
def get_conflict_metrics(input_folder_path: str):
    """
    从指定文件夹路径读取 JSONL 文件，统计指标并返回表格格式
    """
    # 定义各类数据集
    datasets = {
        "academic": ["arxiv"],
        "lifestyle": ["manual"],
        "web": ["web", "wiki", "wit"]
    }

    # 获取文件夹下所有 JSONL 文件
    jsonl_files = [f for f in os.listdir(input_folder_path) if f.endswith(".jsonl")]

    # 按文件名去掉后缀，获取文档名（假设文件名格式为 <dataset>_xxx.jsonl）
    jsonl_doc_names = [f.split("_")[0] for f in jsonl_files]

    # 存储统计结果
    results = {}

    # 遍历每个大类
    for category, subdatasets in datasets.items():
        # 检查该类别下所有子数据集文件是否都存在
        if all(sub in jsonl_doc_names for sub in subdatasets):
            # 收集所有指标
            metrics_accumulator = defaultdict(list)
            total_records = 0

            for sub in subdatasets:
                # 找到对应的 JSONL 文件
                file_name = next(f for f in jsonl_files if f.startswith(sub + "_"))
                file_path = os.path.join(input_folder_path, file_name)

                with open(file_path, "r", encoding="utf-8") as f:
                    for line in f:
                        record = json.loads(line.strip())
                        conflicts = record.get("conflicts", [])
                        set_nums = 0
                        set_nums_text_agent = 0
                        set_nums_visual_agent = 0
                        order_nums = 0
                        ans_img_set = set(re.findall(r'<img_?\d+>', record['final_output']))
                        text_img_in_ans = 0
                        visual_img_in_ans = 0
                        for conflict in conflicts:
                            if conflict["conflict_type"] == "set_conflict":
                                set_nums_text_agent += len(conflict["conflict_info"]["images_only_in_text_agent_response"])
                                set_nums_visual_agent += len(conflict["conflict_info"]["images_only_in_visual_agent_response"])
                                set_nums += (set_nums_text_agent + set_nums_visual_agent)
                                text_img_in_ans += len(set(conflict["conflict_info"]["images_only_in_text_agent_response"]) & ans_img_set)
                                visual_img_in_ans += len(set(conflict["conflict_info"]["images_only_in_visual_agent_response"]) & ans_img_set)
                                
                            elif conflict["conflict_type"] == "order_conflict":
                                order_nums += 1
                        metrics_accumulator["set_nums_text_agent"].append(set_nums_text_agent)
                        metrics_accumulator["set_nums_visual_agent"].append(set_nums_visual_agent)
                        metrics_accumulator["set_nums"].append(set_nums)
                        metrics_accumulator["order_nums"].append(order_nums)
                        metrics_accumulator["text_img_in_ans"].append(text_img_in_ans)
                        metrics_accumulator["visual_img_in_ans"].append(visual_img_in_ans)
                        total_records += 1
            # 数值乘100，保留两位小数
            category_metrics = {metric: (sum(values)/len(values)) for metric, values in metrics_accumulator.items()}
                
            results[category] = category_metrics
    
    return results


In [35]:
input_folder_path = "outputs/gpt-4o-mini/v2/full_img/wo_clip"
input_folder_path2 = "outputs/gpt-4o-mini/v2/"
get_conflict_metrics(input_folder_path2)

{'academic': {'set_nums_text_agent': 0.13636363636363635,
  'set_nums_visual_agent': 0.7373737373737373,
  'set_nums': 0.8737373737373737,
  'order_nums': 0.10101010101010101,
  'text_img_in_ans': 0.12121212121212122,
  'visual_img_in_ans': 0.050505050505050504},
 'lifestyle': {'set_nums_text_agent': 1.0829015544041452,
  'set_nums_visual_agent': 1.3445595854922279,
  'set_nums': 2.4274611398963732,
  'order_nums': 0.17357512953367876,
  'text_img_in_ans': 0.7357512953367875,
  'visual_img_in_ans': 0.23056994818652848},
 'web': {'set_nums_text_agent': 0.05898720089037284,
  'set_nums_visual_agent': 0.11129660545353366,
  'set_nums': 0.17028380634390652,
  'order_nums': 0.03672787979966611,
  'text_img_in_ans': 0.05564830272676683,
  'visual_img_in_ans': 0.0027824151363383415}}

In [36]:
import re
def calculate_total_tokens(log_file):
    total_tokens_sum = 0
    pattern = r'total_tokens=(\d+)'
    max_tokens = 0
    req_nums = 0
    try:
        with open(log_file, 'r', encoding='utf-8') as f:
            for line in f:
                # 查找包含 total_tokens 的行
                if 'total_tokens' in line:
                    # 提取 total_tokens 的值
                    match = re.search(pattern, line)
                    if match:
                        total_tokens = int(match.group(1))
                        req_nums += 1
                        if total_tokens > max_tokens:
                            max_tokens = total_tokens
                        total_tokens_sum += total_tokens
                        # print(f"Found: {total_tokens} tokens")
                        
        print(f"\nTotal token usage: {total_tokens_sum / 168}")
        print(f"Total requests: {req_nums}")
        print(f"Max token usage: {max_tokens}")
        
        return total_tokens_sum
        
    except FileNotFoundError:
        print(f"Error: File '{log_file}' not found")
        return 0
    except Exception as e:
        print(f"Error: {e}")
        return 0

In [37]:
calculate_total_tokens("outputs/gpt-4o-mini/v2/done/logs/manual_T-gpt-4o-mini_V-gpt-4o-mini_J-gpt-4o-mini.log")


Total token usage: 527187.7261904762
Total requests: 2422
Max token usage: 97796


88567538

In [ ]:
with 